# Zotero database cache based on pyzotero.


In [5]:
import pathlib as pl
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import sys
from urllib.parse import urlparse, urlunparse
from icecream import ic

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import re

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:

import pickle
import struct
from pyzotero import zotero

class ZoteroCache:
    HEADER_FORMAT = "I"  # Format for an unsigned integer (serial number)
    HEADER_SIZE = struct.calcsize(HEADER_FORMAT)

    def __init__(self, filename, library_id, library_type, api_key):
        """
        Initialize the ZoteroCache with the given filename and Zotero API credentials.
        """
        self.filename = filename
        self.zot = zotero.Zotero(library_id, library_type, api_key)

    def write_cache(self):
        """
        Fetch data from Zotero and write it to the cache file along with the current version.
        """
        # Fetch data from Zotero
        parent_items = self.zot.everything(self.zot.top())
        current_version = self.zot.last_modified_version()

        with open(self.filename, 'wb') as f:
            # Write the current version as a fixed-size header
            f.write(struct.pack(self.HEADER_FORMAT, current_version))
            # Serialize and write the parent items
            pickle.dump(parent_items, f)

    def read_version(self):
        """
        Read only the version number from the cache file.
        """
        try:
            with open(self.filename, 'rb') as f:
                # Read and unpack the fixed-size header
                header_data = f.read(self.HEADER_SIZE)
                return struct.unpack(self.HEADER_FORMAT, header_data)[0]
        except FileNotFoundError:
            return None  # Cache file does not exist

    def read_cache(self):
        """
        Read the cached data from the file.
        """
        try:
            with open(self.filename, 'rb') as f:
                # Skip the fixed-size header
                f.seek(self.HEADER_SIZE)
                # Load and deserialize the parent items
                return pickle.load(f)
        except FileNotFoundError:
            return None  # Cache file does not exist

    def is_cache_valid(self):
        """
        Check if the cached version matches the current version from Zotero.
        """
        cached_version = self.read_version()
        current_version = self.zot.last_modified_version()
        return cached_version == current_version

    def get_data(self):
        """
        Retrieve data from cache if valid; otherwise update cache and fetch fresh data.
        """
        if self.is_cache_valid():
            print("Cache is valid. Reading data from cache.")
            return self.read_cache()
        else:
            print("Cache is outdated or missing. Fetching fresh data.")
            self.write_cache()
            return self.read_cache()

In [6]:
# Replace these with your actual Zotero API credentials
library_id = "your_library_id"
library_type = "your_library_type"  # e.g., 'user' or 'group'
api_key = "your_api_key"

cache_filename = "zotero_cache.bin"

zotero_cache = ZoteroCache(rfw.zoterodb_cache_file, rfw.library_id, rfw.library_type, rfw.api_key)

# Retrieve data (from cache if valid or fresh otherwise)
data = zotero_cache.get_data()

print("Retrieved Data:", data[:5])  # Print first 5 items for brevity


AttributeError: module 'refwrangle' has no attribute 'zoterodb_cache_file'